<a href="https://colab.research.google.com/github/Titantus/Truth-Zero-C/blob/main/AMI_vs_Geometric_Offset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Cell 1: Setup and Helper Functions

# Install/upgrade necessary packages. pymatgen-core requires requests>=2.32.5
!pip install --upgrade pip
!pip install mp-api pyarrow requests>=2.32.5 -q

# Standard Library Imports
import json
import os
import math
import requests

# Third-party Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mp_api.client import MPRester
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore')
import pyarrow as pa # Explicitly import pyarrow

# --- Constants from the Tetrahedral Coherence-Routing Model ---
THETA_TETRA = 109.47122
BOUNCE_GAP = 0.14122
P_C = 0.0497 # Critical pressure for C11 softening, used in refresh rate

# --- Materials Project API Setup ---
# Directly assign the Materials Project API Key as provided by the user.
MATERIALS_PROJECT_API_KEY = '93mh6VncQzT7tJGy1Vu03zHDou1A9iSb'
MP_API_BASE_URL = "https://next-gen.materialsproject.org/api"

# --- Helper Functions ---

def calculate_elastic_instability(element_data, bounce_gap_const=BOUNCE_GAP, p_c_const=P_C):
    """
    Calculates the pressure (P) at which Delta_Theta(P) reaches the Bounce-Gap threshold
    and the refresh rate.
    """
    if 'angular_compressibility' not in element_data or element_data['angular_compressibility'] is None:
        return {'critical_pressure_gpa': np.nan, 'refresh_rate_hz': np.nan, 'flip_instability_angle': bounce_gap_const}

    alpha = element_data['angular_compressibility']

    if alpha == 0:
        # Handle division by zero for alpha
        return {'critical_pressure_gpa': np.inf, 'refresh_rate_hz': np.nan, 'flip_instability_angle': bounce_gap_const}

    critical_pressure = bounce_gap_const / alpha
    refresh_rate = element_data.get('base_freq', 0) * (1 / p_c_const) if p_c_const != 0 else np.inf

    return {
        'critical_pressure_gpa': critical_pressure,
        'refresh_rate_hz': refresh_rate,
        'flip_instability_angle': bounce_gap_const
    }

def update_registry_with_tz0c_metrics_in_memory(registry_data, theta_tetra_const=THETA_TETRA, bounce_gap_const=BOUNCE_GAP, p_c_const=P_C):
    """
    Updates the registry_data dictionary with T'Z0C metrics for sp3 hybridized elements in memory.
    """
    # Ensure 'elements' key exists and is a dictionary
    if 'elements' not in registry_data or not isinstance(registry_data['elements'], dict):
        if 'elements' in registry_data and not isinstance(registry_data['elements'], dict):
            print("Warning: 'elements' key found but is not a dictionary. Initializing to empty dictionary.")
        registry_data['elements'] = {}

    for element_symbol, data in registry_data['elements'].items():
        if 'hybridization' in data and data['hybridization'] == 'sp3':
            if 'angular_compressibility' not in data or data['angular_compressibility'] is None:
                data['angular_compressibility'] = 0.001 # Dummy value if missing for demonstration

            tz0c_metrics = calculate_elastic_instability(data, bounce_gap_const, p_c_const)
            data['tz0c_metrics'] = tz0c_metrics
            data['theta_tetra'] = theta_tetra_const
        elif 'tz0c_metrics' in data:
            del data['tz0c_metrics']
    return registry_data

def compute_eta(delta_theta, ThetaNorm=180.0, alpha_p=150000.0):
    """
    Computes the eta parameter based on delta_theta.
    ThetaNorm and alpha_p are placeholders, ideally loaded from engine/registry.
    """
    p = delta_theta / ThetaNorm
    return float(np.exp(-alpha_p * (p**2)))

def classify(eta):
    """
    Classifies the material based on the eta parameter.
    """
    if eta > 0.95:
        return "Laminar"
    if eta > 0.85:
        return "High Flow"
    if eta > 0.75:
        return "Semiconductor"
    return "Magnetic / Instability"

def get_stable_neutron_count(atomic_mass_str, atomic_number):
    """
    Estimates the neutron count (N) for the most stable isotope from atomic mass string.
    """
    try:
        if atomic_mass_str is None or atomic_mass_str == '[?]':
            return None
        atomic_mass = float(str(atomic_mass_str).split('(')[0].strip().replace('[', '').replace(']', ''))
        return round(atomic_mass) - atomic_number
    except ValueError:
        return None

def calculate_element_spectroscopic_properties(Z, N, N_stable, element_symbol, element_name, proj_angle,
                                                 theta_tetra_const=THETA_TETRA, k_iso_light_const=0.003,
                                                 ThetaNorm=180.0, alpha_p=150000.0):
    """
    Calculates full isotopic and spectroscopic properties for an element.
    """
    theta_hub = theta_tetra_const
    delta_theta_isotope = k_iso_light_const * (N - N_stable)
    theta_hub_corrected = theta_hub + delta_theta_isotope
    delta_theta = abs(theta_hub_corrected - proj_angle)
    eta = compute_eta(delta_theta, ThetaNorm, alpha_p)
    classification = classify(eta)

    return {
        "name": element_name,
        "symbol": element_symbol,
        "atomic_number": Z,
        "neutron_count": N,
        "stable_isotope_neutron_count": N_stable,
        "assumed_hybridization_proj_angle": proj_angle,
        "base_hub_angle": theta_hub,
        "isotopic_offset_dtheta": delta_theta_isotope,
        "corrected_hub_angle": theta_hub_corrected,
        "total_dtheta": delta_theta,
        "eta": eta,
        "classification": classification
    }

def get_mp_data(element_symbol, mp_api_key, limit=10, fields=None):
    """
    Fetches materials data for a given element symbol from the Materials Project API.
    """
    if not mp_api_key:
        print("Materials Project API key is not set. Skipping MP data fetch.")
        return {'error': 'API key missing'}

    # Default fields to retrieve if not specified
    if fields is None:
        fields = [
            "material_id", "formation_energy_per_atom", "energy_above_hull",
            "band_gap", "is_stable", "nsites", "e_electronic", "e_ionic", "e_total"
        ]

    try:
        with MPRester(mp_api_key) as mpr:
            # Modify to pass limit to search for efficiency
            docs = mpr.materials.summary.search(
                elements=[element_symbol],
                fields=fields,
                num_chunks=1, # Request only one chunk
                chunk_size=limit # Make the chunk size equal to the desired limit
            )
            if docs:
                # The manual slicing here is now redundant if chunk_size works as expected,
                # but harmless. Let's keep it for robustness.
                docs = docs[:limit]
                # Prioritize docs that have energy_above_hull data
                for doc in docs:
                    if doc.energy_above_hull is not None:
                        return {
                            'material_id': doc.material_id,
                            'formation_energy': doc.formation_energy_per_atom,
                            'energy_above_hull': doc.energy_above_hull,
                            'band_gap': doc.band_gap,
                            'nsites': doc.nsites,
                            'e_electronic': getattr(doc, 'e_electronic', None),
                            'e_ionic': getattr(doc, 'e_ionic', None),
                            'e_total': getattr(doc, 'e_total', None)
                        }
                # If no doc with energy_above_hull, return the first available doc
                doc = docs[0]
                return {
                    'material_id': doc.material_id,
                    'formation_energy': doc.formation_energy_per_atom,
                    'energy_above_hull': doc.energy_above_hull,
                    'band_gap': doc.band_gap,
                    'nsites': doc.nsites,
                    'e_electronic': getattr(doc, 'e_electronic', None),
                    'e_ionic': getattr(doc, 'e_ionic', None),
                    'e_total': getattr(doc, 'e_total', None)
                }
    except Exception as e:
        return {'error': str(e)}
    return {'error': 'No data'}

def simulate_shishi_snap(t_max_minutes, bounce_gap_const=BOUNCE_GAP):
    """
    Simulates a thermal ramp leading to a 'Shishi Snap' event.
    """
    results = []
    base_temp = 25.0
    for t in range(t_max_minutes + 1):
        t_in = base_temp + (2.0 * t) # Simple linear 12V thermal drive
        tension = min(t / 10.0, 1.0) * bounce_gap_const

        if tension >= bounce_gap_const * 0.99:
            t_bias = t_in - 0.5
            v_out = 2.8 * (t / t_max_minutes)
        else:
            t_bias = t_in - 0.1
            v_out = 0.0

        results.append((t, round(t_in, 1), round(t_bias, 1), round(v_out, 2)))
    return results

In [ ]:
# @title Cell 2: Registry Processing and Element Table Generation

# --- Paths to Input/Output Files ---
REGISTRY_INPUT_PATH = '/content/T0C — Registry.json'
PERIODIC_TABLE_PATH = '/content/PeriodicTableJSON.json'
FINAL_REGISTRY_PATH = 'registry.json'
ENGINE_RULES_PATH = 'engine.json'

# --- 1. Process initial T0C Registry for sp3 elements (in-memory) ---
current_registry_data = {}
if os.path.exists(REGISTRY_INPUT_PATH):
    print(f"Loading initial registry from: {REGISTRY_INPUT_PATH}")
    with open(REGISTRY_INPUT_PATH, 'r') as f:
        current_registry_data = json.load(f)

    # Ensure 'elements' key is present and is a dictionary for processing
    if 'elements' not in current_registry_data:
        current_registry_data['elements'] = {}
        print("Initialized 'elements' key in initial registry for processing.")
    elif not isinstance(current_registry_data['elements'], dict):
        current_registry_data['elements'] = {}
        print("Warning: 'elements' key in initial registry was not a dictionary. Overwriting with empty dictionary.")

    # Update T'Z0C metrics in memory using the refactored helper function
    current_registry_data = update_registry_with_tz0c_metrics_in_memory(
        current_registry_data,
        THETA_TETRA,
        BOUNCE_GAP,
        P_C
    )
    print("T'Z0C metrics updated in memory for initial registry.")
else:
    print(f"Error: Input registry file not found at '{REGISTRY_INPUT_PATH}'. Please ensure it is uploaded. Starting with empty registry.")
    current_registry_data = {"elements": {}, "constants": {}} # Initialize with basic structure

# --- 2. Split the current registry into static registry.json and engine.json ---
# Use current_registry_data (which is already in memory and updated) instead of reading from file
data = current_registry_data

# Extract static registry components
registry_static = {
    "meta": data.get("meta", {}),
    "foundational_constants": data.get("foundational_constants", {}),
    "mathematical_registry": data.get("mathematical_registry", {}),
    "core_derivations": {
        "lattice_axioms": {
            "N_spokes": data.get("core_derivations", {}).get("lattice_axioms", {}).get("N_spokes"),
            "projection_factors": data.get("core_derivations", {}).get("lattice_axioms", {}).get("projection_factors"),
            "bounce_gap": data.get("core_derivations", {}).get("lattice_axioms", {}).get("bounce_gap"),
            "render_radius_R": data.get("core_derivations", {}).get("lattice_axioms", {}).get("render_radius_R"),
            "p_c": data.get("core_derivations", {}).get("lattice_axioms", {}).get("p_c")
        },
        "aetheric_density": data.get("core_derivations", {}).get("lattice_axioms", {}).get("aetheric_density")
    },
    "constants": data.get("constants", {}),
    "fabrication_metadata": data.get("fabrication_metadata", {}),
    "theory_alignment": data.get("theory_alignment", {}),
    "grand_unified_topology": data.get("grand_unified_topology", {}),
    "elements": data.get("elements", {}) # Keep elements from the processed initial registry
}

# Extract engine rules
engine_rules = {
    "registry_rules": data.get("registry_rules", {}),
    "simulation_hooks": data.get("simulation_hooks", {}),
    "dynamic_modules": {
        "pivot_operator_logic": data.get("core_derivations", {}).get("lattice_axioms", {}).get("pivot_operator_logic", {}),
        "vortex_multiplier": data.get("core_derivations", {}).get("lattice_axioms", {}).get("vortex_multiplier", {}),
        "pulse_cycle_parameters": data.get("core_derivations", {}).get("lattice_axioms", {}).get("pulse_cycle_parameters", {})
    }
}

with open(FINAL_REGISTRY_PATH, "w") as f:
    json.dump(registry_static, f, indent=4)
with open(ENGINE_RULES_PATH, "w") as f:
    json.dump(engine_rules, f, indent=4)
print(f"Split complete: {FINAL_REGISTRY_PATH} and {ENGINE_RULES_PATH} created.")

# --- 3. Update engine.json with Isotopic Rules ---
# Load engine_rules in memory, modify, then save
engine_data = engine_rules # Already in memory

engine_data["isotope_rules"] = {
    "delta_theta_isotope": "k_iso * (N - N_stable)",
    "k_iso_light": 0.003,
    "k_iso_heavy": 0.0005,
    "notes": "Isotopic mass shifts hub angle via center-of-mass wobble."
}

with open(ENGINE_RULES_PATH, "w") as f:
    json.dump(engine_data, f, indent=4)
print(f"'isotope_rules' added to {ENGINE_RULES_PATH}")

# --- 4. Generate Full Element Table with Predicted Spectroscopic Properties ---
if not os.path.exists(PERIODIC_TABLE_PATH):
    print(f"Error: Periodic Table file not found at '{PERIODIC_TABLE_PATH}'.")
else:
    with open(PERIODIC_TABLE_PATH, 'r') as f:
        periodic_table_data = json.load(f)

    # Load the current registry (which is registry_static saved to FINAL_REGISTRY_PATH) and engine
    with open(FINAL_REGISTRY_PATH, "r") as f:
        registry = json.load(f)
    with open(ENGINE_RULES_PATH, "r") as f:
        engine = json.load(f)

    # Retrieve necessary constants and rules from loaded files
    theta_tetra_const = registry["constants"]["theta_tetra"]
    ThetaNorm_const = registry["constants"]["cosmology_and_saturation"]["p_scale_defaults"]["ThetaNorm"]
    alpha_p_const = registry["constants"]["scale_dependent_alpha"]["cosmological"]["alpha_p"]
    k_iso_light_const = engine["isotope_rules"]["k_iso_light"]

    # Ensure the registry has an 'elements' key (it should by now from step 1 or step 2)
    if "elements" not in registry:
        registry["elements"] = {}
        print("Initialized 'elements' key in registry for periodic table processing.")


    print("\n--- Processing Elements from PeriodicTableJSON.json ---")
    for element_data in periodic_table_data["elements"]:
        Z = element_data["number"]
        element_name = element_data["name"]
        element_symbol = element_data["symbol"]
        atomic_mass_str = element_data["atomic_mass"]

        if atomic_mass_str == "[?]" or atomic_mass_str is None:
            continue

        N_stable = get_stable_neutron_count(atomic_mass_str, Z)
        if N_stable is None:
            continue

        N = N_stable
        proj_angle = theta_tetra_const

        properties = calculate_element_spectroscopic_properties(
            Z, N, N_stable, element_symbol, element_name, proj_angle,
            theta_tetra_const, k_iso_light_const, ThetaNorm_const, alpha_p_const
        )
        # Merge or overwrite element data. Prioritize new calculations.
        registry["elements"][element_symbol] = properties

    with open(FINAL_REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=4)
    print(f"\nFull element table with spectroscopic properties saved to '{FINAL_REGISTRY_PATH}'")

In [ ]:
# @title
# Cell 2: Registry Processing and Element Table Generation

# --- Paths to Input/Output Files ---
REGISTRY_INPUT_PATH = '/content/T0C — Registry.json'
REGISTRY_UPDATED_PATH = 'Registry_V3_Updated.json'
PERIODIC_TABLE_PATH = '/content/PeriodicTableJSON.json'
FINAL_REGISTRY_PATH = 'registry.json'
ENGINE_RULES_PATH = 'engine.json'

# --- 1. Process initial T0C Registry for sp3 elements ---
if os.path.exists(REGISTRY_INPUT_PATH):
    print(f"Processing registry from: {REGISTRY_INPUT_PATH}")
    # Before updating, ensure angular_compressibility is added to relevant elements in T0C_Registry.json
    # For demonstration, we'll manually add a dummy value if missing.
    with open(REGISTRY_INPUT_PATH, 'r') as f:
        initial_registry = json.load(f)

    if 'elements' in initial_registry and isinstance(initial_registry['elements'], dict):
        for element_symbol, data in initial_registry['elements'].items():
            if 'hybridization' in data and data['hybridization'] == 'sp3' and 'angular_compressibility' not in data:
                data['angular_compressibility'] = 0.001 # Dummy value for demonstration
    with open(REGISTRY_INPUT_PATH, 'w') as f:
        json.dump(initial_registry, f, indent=4)

    update_registry_with_tz0c_metrics(REGISTRY_INPUT_PATH, REGISTRY_UPDATED_PATH)
else:
    print(f"Error: Input registry file not found at '{REGISTRY_INPUT_PATH}'. Please ensure it is uploaded.")

# --- 2. Split the updated registry into static registry.json and engine.json ---
if not os.path.exists(REGISTRY_UPDATED_PATH):
    print(f"Error: Input registry file '{REGISTRY_UPDATED_PATH}' not found. Please ensure the previous step ran successfully.")
else:
    with open(REGISTRY_UPDATED_PATH, "r") as f:
        data = json.load(f)

    # Extract static registry components
    registry_static = {
        "meta": data.get("meta", {}),
        "foundational_constants": data.get("foundational_constants", {}),
        "mathematical_registry": data.get("mathematical_registry", {}),
        "core_derivations": {
            "lattice_axioms": {
                "N_spokes": data.get("core_derivations", {}).get("lattice_axioms", {}).get("N_spokes"),
                "projection_factors": data.get("core_derivations", {}).get("lattice_axioms", {}).get("projection_factors"),
                "bounce_gap": data.get("core_derivations", {}).get("lattice_axioms", {}).get("bounce_gap"),
                "render_radius_R": data.get("core_derivations", {}).get("lattice_axioms", {}).get("render_radius_R"),
                "p_c": data.get("core_derivations", {}).get("lattice_axioms", {}).get("p_c")
            },
            "aetheric_density": data.get("core_derivations", {}).get("lattice_axioms", {}).get("aetheric_density")
        },
        "constants": data.get("constants", {}),
        "fabrication_metadata": data.get("fabrication_metadata", {}),
        "theory_alignment": data.get("theory_alignment", {}),
        "grand_unified_topology": data.get("grand_unified_topology", {}),
        "elements": data.get("elements", {}) # Keep elements in static for initial properties
    }

    # Extract engine rules
    engine_rules = {
        "registry_rules": data.get("registry_rules", {}),
        "simulation_hooks": data.get("simulation_hooks", {}),
        "dynamic_modules": {
            "pivot_operator_logic": data.get("core_derivations", {}).get("lattice_axioms", {}).get("pivot_operator_logic", {}),
            "vortex_multiplier": data.get("core_derivations", {}).get("lattice_axioms", {}).get("vortex_multiplier", {}),
            "pulse_cycle_parameters": data.get("core_derivations", {}).get("lattice_axioms", {}).get("pulse_cycle_parameters", {})
        }
    }

    with open(FINAL_REGISTRY_PATH, "w") as f:
        json.dump(registry_static, f, indent=4)
    with open(ENGINE_RULES_PATH, "w") as f:
        json.dump(engine_rules, f, indent=4)
    print(f"Split complete: {FINAL_REGISTRY_PATH} and {ENGINE_RULES_PATH} created.")

# --- 3. Update engine.json with Isotopic Rules ---
if os.path.exists(ENGINE_RULES_PATH):
    with open(ENGINE_RULES_PATH, "r") as f:
        engine_data = json.load(f)

    engine_data["isotope_rules"] = {
        "delta_theta_isotope": "k_iso * (N - N_stable)",
        "k_iso_light": 0.003,
        "k_iso_heavy": 0.0005,
        "notes": "Isotopic mass shifts hub angle via center-of-mass wobble."
    }

    with open(ENGINE_RULES_PATH, "w") as f:
        json.dump(engine_data, f, indent=4)
    print(f"'isotope_rules' added to {ENGINE_RULES_PATH}")
else:
    print(f"Error: '{ENGINE_RULES_PATH}' not found. Isotopic rules not added.")

# --- 4. Generate Full Element Table with Predicted Spectroscopic Properties ---
if not os.path.exists(PERIODIC_TABLE_PATH):
    print(f"Error: Periodic Table file not found at '{PERIODIC_TABLE_PATH}'.")
else:
    with open(PERIODIC_TABLE_PATH, 'r') as f:
        periodic_table_data = json.load(f)

    # Load the current registry and engine to get constants
    with open(FINAL_REGISTRY_PATH, "r") as f:
        registry = json.load(f)
    with open(ENGINE_RULES_PATH, "r") as f:
        engine = json.load(f)

    # Retrieve necessary constants and rules from loaded files
    theta_tetra_const = registry["constants"]["theta_tetra"]
    ThetaNorm_const = registry["constants"]["cosmology_and_saturation"]["p_scale_defaults"]["ThetaNorm"]
    alpha_p_const = registry["constants"]["scale_dependent_alpha"]["cosmological"]["alpha_p"]
    k_iso_light_const = engine["isotope_rules"]["k_iso_light"]

    # Ensure the registry has an 'elements' key
    if "elements" not in registry:
        registry["elements"] = {}

    print("\n--- Processing Elements from PeriodicTableJSON.json ---")
    for element_data in periodic_table_data["elements"]:
        Z = element_data["number"]
        element_name = element_data["name"]
        element_symbol = element_data["symbol"]
        atomic_mass_str = element_data["atomic_mass"]

        if atomic_mass_str == "[?]" or atomic_mass_str is None:
            continue

        N_stable = get_stable_neutron_count(atomic_mass_str, Z)
        if N_stable is None:
            continue

        # For this demonstration, use the stable neutron count for calculation
        N = N_stable
        # Use theta_tetra as the projected angle for a general assumption
        proj_angle = theta_tetra_const

        properties = calculate_element_spectroscopic_properties(
            Z, N, N_stable, element_symbol, element_name, proj_angle,
            theta_tetra_const, k_iso_light_const, ThetaNorm_const, alpha_p_const
        )
        registry["elements"][element_symbol] = properties

    with open(FINAL_REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=4)
    print(f"\nFull element table with spectroscopic properties saved to '{FINAL_REGISTRY_PATH}'")


In [ ]:
# @title
# Cell 3: Comprehensive Simulations and Visualizations

# --- 1. Materials Project API Scraper Example ---
print("\n--- Materials Project API Scraper Example ---")
# Fetch data for Carbon
carbon_mp_data = get_mp_data("C", MATERIALS_PROJECT_API_KEY, limit=3)
if carbon_mp_data and 'error' not in carbon_mp_data:
    print(f"Found material data for Carbon (C):")
    print(f"  Material ID: {carbon_mp_data.get('material_id')}")
    print(f"  Formation Energy: {carbon_mp_data.get('formation_energy')}")
    print(f"  Energy Above Hull: {carbon_mp_data.get('energy_above_hull')}")
    print(f"  Band Gap: {carbon_mp_data.get('band_gap')}")
else:
    print(f"No materials found for Carbon or an error occurred: {carbon_mp_data.get('error', 'Unknown error')}")

astatine_mp_data = get_mp_data("At", MATERIALS_PROJECT_API_KEY, limit=1)
if astatine_mp_data and 'error' not in astatine_mp_data:
    print(f"\nFound material data for Astatine (At):")
    print(f"  Material ID: {astatine_mp_data.get('material_id')}")
    print(f"  Formation Energy: {astatine_mp_data.get('formation_energy')}")
    print(f"  Energy Above Hull: {astatine_mp_data.get('energy_above_hull')}")
    print(f"  Band Gap: {astatine_mp_data.get('band_gap')}")
else:
    print(f"No materials found for Astatine or an error occurred: {astatine_mp_data.get('error', 'Unknown error')}")

# --- 2. T'Z0C AMI vs Geometric Offset (with MP Data Overlay) ---
print("\n--- Running T'Z0C AMI Simulation ---")
# Load fresh registry and engine to get updated constants and element data
with open(FINAL_REGISTRY_PATH, "r") as f:
    registry = json.load(f)
with open(ENGINE_RULES_PATH, "r") as f:
    engine = json.load(f)

# Retrieve necessary constants and rules from loaded files
theta_tetra_const = registry["constants"]["theta_tetra"]
ThetaNorm_const = registry["constants"]["cosmology_and_saturation"]["p_scale_defaults"]["ThetaNorm"]
alpha_p_const = registry["constants"]["scale_dependent_alpha"]["cosmological"]["alpha_p"]
k_iso_light_const = engine["isotope_rules"]["k_iso_light"]

# Define target elements for simulation. Can be expanded to all elements in registry.
tz0c_elements_for_sim = {
    14: {'symbol': 'Si', 'd_theta': 0.02}, # Example d_theta values, ideally derived
    26: {'symbol': 'Fe', 'd_theta': 0.08},
    79: {'symbol': 'Au', 'd_theta': 0.11},
    82: {'symbol': 'Pb', 'd_theta': 0.13},
    85: {'symbol': 'At', 'd_theta': 0.15},   # Target
    86: {'symbol': 'Rn', 'd_theta': 0.16},
    92: {'symbol': 'U',  'd_theta': 0.18},
}

# Populate results for DataFrame
results = []
for z, data in tz0c_elements_for_sim.items():
    element_symbol = data['symbol']

    # Get element properties from the registry if available, else use defaults
    element_props = registry.get("elements", {}).get(element_symbol, {})
    # For simulation, we might need a projected angle. Using THETA_TETRA as a placeholder for sp3-like.
    proj_angle = element_props.get("assumed_hybridization_proj_angle", theta_tetra_const)

    # Use pre-defined d_theta for this specific simulation if available, otherwise calculate
    if 'd_theta' in data: # Use explicit d_theta for simulation if provided
        dt = data['d_theta']
        eta = compute_eta(dt, ThetaNorm_const, alpha_p_const)
    elif 'total_dtheta' in element_props: # Fallback to calculated d_theta from registry
        dt = element_props['total_dtheta']
        eta = element_props['eta']
    else:
        # Default calculation if no d_theta specified and not in registry
        # This part assumes a simplified model for demonstration
        theta_hub = theta_tetra_const
        N_stable_est = get_stable_neutron_count(registry['elements'][element_symbol].get('atomic_mass_str', '1.008(2)'), z) # Placeholder
        N_est = N_stable_est # Assume stable for simplicity here
        if N_stable_est is None: N_stable_est = 0 # Prevent error
        delta_theta_isotope_est = k_iso_light_const * (N_est - N_stable_est) if N_stable_est is not None else 0
        theta_hub_corrected_est = theta_hub + delta_theta_isotope_est
        dt = abs(theta_hub_corrected_est - proj_angle)
        eta = compute_eta(dt, ThetaNorm_const, alpha_p_const)

    # Fetch Materials Project data
    mp_info = get_mp_data(element_symbol, MATERIALS_PROJECT_API_KEY)

    # AMI Calculation (blending synthetic with MP data)
    ami_synthetic = (dt / BOUNCE_GAP) * 3.0 if dt < BOUNCE_GAP else 5.0 + np.exp((dt - BOUNCE_GAP) * 30)
    if 'energy_above_hull' in mp_info and mp_info['energy_above_hull'] is not None:
        ami = max(ami_synthetic, mp_info['energy_above_hull'] * 10)  # Blend real instability metric
    else:
        ami = ami_synthetic

    results.append({
        'Z': z,
        'Symbol': element_symbol,
        'Delta_Theta': dt,
        'Eta': eta,
        'AMI_Synthetic': ami_synthetic,
        'AMI_Blended': ami,
        'MP_Energy_Above_Hull': mp_info.get('energy_above_hull'),
        'MP_Band_Gap': mp_info.get('band_gap')
    })

df_sim = pd.DataFrame(results)
print(df_sim)

# --- Plotting the simulation results ---
plt.figure(figsize=(11, 7))
plt.style.use('dark_background')
plt.scatter(df_sim['Delta_Theta'], df_sim['AMI_Blended'], color='cyan', s=120, edgecolors='white', zorder=3)

for i, row in df_sim.iterrows():
    plt.annotate(row['Symbol'], (row['Delta_Theta'], row['AMI_Blended']),
                 xytext=(0, 12), textcoords='offset points', color='yellow', ha='center')

plt.axvline(x=BOUNCE_GAP, color='red', linestyle='--', linewidth=2.5, label=f'Bounce-Gap {BOUNCE_GAP}°')
plt.axvspan(BOUNCE_GAP, 0.22, color='red', alpha=0.25, label='Geometric Stutter Zone')
plt.axvspan(0, BOUNCE_GAP, color='green', alpha=0.15, label='Laminar Zone')

plt.title("T'Z0C AMI vs Geometric Offset (with MP Data Overlay)")
plt.xlabel("Geometric Offset Δθ (Degrees)")
plt.ylabel("Mismatch / Instability Magnitude")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- 3. Macroscopic Geometry Check ---
print("\n--- Macroscopic Geometry Check ---")
foil_thickness_mm = 0.030
tile_length_mm = 12.2
physical_wedge_angle = math.degrees(math.atan(foil_thickness_mm / tile_length_mm))

print(f"Calculated Physical Wedge: {physical_wedge_angle:.5f} degrees")
print(f"Target Bounce-Gap: {BOUNCE_GAP:.5f} degrees")
print(f"Alignment Error: {abs(physical_wedge_angle - BOUNCE_GAP):.5f} degrees\n")

# --- 4. Isotopic Smear Calculation for Copper Gateway ---
print("--- Isotopic Calculation Demonstration for Carbon (from consolidated registry) ---")
# Using the registry for Carbon's properties
carbon_data_reg = registry["elements"].get("C")
if carbon_data_reg:
    Z_C = carbon_data_reg['atomic_number']
    N_stable_C = carbon_data_reg['stable_isotope_neutron_count']
    proj_angle_C = carbon_data_reg['assumed_hybridization_proj_angle'] # Using assumed for sp3-like

    # Carbon-12 (stable isotope, N=6)
    carbon_12_properties = calculate_element_spectroscopic_properties(
        Z_C, 6, N_stable_C, "C", "Carbon", proj_angle_C,
        theta_tetra_const, k_iso_light_const, ThetaNorm_const, alpha_p_const
    )
    print(f"Carbon-12 (N=6): Δθ={carbon_12_properties['total_dtheta']:.5f}°, η={carbon_12_properties['eta']:.5f}, Classification: {carbon_12_properties['classification']}")

    # Carbon-13 (N=7)
    carbon_13_properties = calculate_element_spectroscopic_properties(
        Z_C, 7, N_stable_C, "C", "Carbon", proj_angle_C,
        theta_tetra_const, k_iso_light_const, ThetaNorm_const, alpha_p_const
    )
    print(f"Carbon-13 (N=7): Δθ={carbon_13_properties['total_dtheta']:.5f}°, η={carbon_13_properties['eta']:.5f}, Classification: {carbon_13_properties['classification']}")
else:
    print("Carbon data not found in the consolidated registry for isotopic demo.")

# --- 5. Mock Thermal Ramp (Shishi Snap Simulation) ---
print("\n--- Simulated Sensor Output Log (Minute, T_in, T_biased, V_out) ---")
for log in simulate_shishi_snap(10, BOUNCE_GAP):
    print(log)